In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("dirty_cafe_sales.csv")

In [3]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [4]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [5]:
df.shape

(10000, 8)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [7]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB
None


In [8]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

Shape: (10000, 8)

Columns:
['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']

Missing Values:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Duplicate Rows:
0

Data Types:
Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object


## 1. Data Quality Report

The dataset contains 10,000 rows and 8 columns. Initial inspection shows missing values in seven columns. The `Location` column has the highest number of missing values (3,265), followed by `Payment Method` (2,579).

The `Quantity`, `Price Per Unit`, and `Total Spent` columns are currently stored as object data types, even though they represent numerical values. The `Transaction Date` column is also stored as an object and requires conversion to a proper datetime format.

No exact duplicate rows were found in the initial dataset. Further cleaning will focus on missing values, inconsistent entries, data type correction, standardization, and outlier detection.

In [9]:
quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Missing Percentage": (df.isnull().sum().values / len(df) * 100).round(2),
    "Unique Values": df.nunique().values
})

quality_report

,Column,Data Type,Missing Values,Missing Percentage,Unique Values
0,Transaction ID,object,0,0.00,10000
1,Item,object,333,3.33,10
2,Quantity,object,138,1.38,7
3,Price Per Unit,object,179,1.79,8
4,Total Spent,object,173,1.73,19
5,Payment Method,object,2579,25.79,5
6,Location,object,3265,32.65,4
7,Transaction Date,object,159,1.59,367


In [10]:
for column in df.columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False).head(20))


--- Transaction ID ---
Transaction ID
TXN_9226047    1
TXN_8567525    1
TXN_4583012    1
TXN_6796890    1
TXN_9933628    1
TXN_4302199    1
TXN_5548914    1
TXN_3528020    1
TXN_9668108    1
TXN_8076061    1
TXN_7936002    1
TXN_3124078    1
TXN_6120851    1
TXN_5762440    1
TXN_9954652    1
TXN_8866974    1
TXN_8927252    1
TXN_1736287    1
TXN_7640952    1
TXN_8467949    1
Name: count, dtype: int64

--- Item ---
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

--- Quantity ---
Quantity
5          2013
2          1974
4          1863
3          1849
1          1822
UNKNOWN     171
ERROR       170
NaN         138
Name: count, dtype: int64

--- Price Per Unit ---
Price Per Unit
3.0        2429
4.0        2331
2.0        1227
5.0        1204
1.0        1143
1.5        1133
ERROR       190
NaN         179
UNKNOWN     164

## 2. Data Cleaning Strategy

The initial inspection revealed several data-quality issues, including missing values, placeholder values such as `UNKNOWN` and `ERROR`, incorrect data types, and inconsistent date entries.

For categorical columns, invalid placeholder values will be converted to missing values and filled using the mode where appropriate. For numerical columns, invalid values will be converted to missing values and filled using the median, which is less affected by extreme values than the mean.

The `Transaction Date` column will be converted from text to datetime format. Duplicate rows were checked during the initial inspection, and no exact duplicate rows were found.

In [11]:
before_cleaning = {
    "Rows": len(df),
    "Columns": len(df.columns),
    "Total Missing Values": df.isnull().sum().sum(),
    "Duplicate Rows": df.duplicated().sum(),
    "Object Columns": (df.dtypes == "object").sum()
}

before_cleaning

{'Rows': 10000,
 'Columns': 8,
 'Total Missing Values': np.int64(6826),
 'Duplicate Rows': np.int64(0),
 'Object Columns': np.int64(8)}

## 3. Handling Invalid Placeholder Values

The dataset contains placeholder values such as `ERROR` and `UNKNOWN`. These values do not represent valid information and can interfere with data analysis. Therefore, they are converted to `NaN` so they can be handled using appropriate missing-value strategies.

In [12]:
# Replace invalid placeholder values with NaN
df = df.replace(["ERROR", "UNKNOWN"], np.nan)

# Check the updated missing values
df.isnull().sum()

,0
Transaction ID,0
Item,969
Quantity,479
Price Per Unit,533
Total Spent,502
Payment Method,3178
Location,3961
Transaction Date,460


In [13]:
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[numeric_columns].dtypes

,0
Quantity,float64
Price Per Unit,float64
Total Spent,float64


In [14]:
df[numeric_columns].isnull().sum()

,0
Quantity,479
Price Per Unit,533
Total Spent,502


## 4. Handling Missing Numerical Values

The `Quantity`, `Price Per Unit`, and `Total Spent` columns contain missing values. Since these columns contain numerical sales data that may include extreme values, the median is used for imputation. The median is less sensitive to outliers than the mean and therefore provides a more robust estimate of a typical value.

Missing values in each numerical column are replaced with the median of that respective column.

In [15]:
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

In [16]:
df[numeric_columns].isnull().sum()

,0
Quantity,0
Price Per Unit,0
Total Spent,0


## 5. Handling Missing Categorical Values

The `Item`, `Payment Method`, and `Location` columns contain missing values. Since these are categorical variables, missing values are filled using the mode, which is the most frequently occurring valid category in each column. This helps preserve the existing distribution of the categorical data.

In [17]:
categorical_columns = ["Item", "Payment Method", "Location"]

for column in categorical_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

In [18]:
df[categorical_columns].isnull().sum()

,0
Item,0
Payment Method,0
Location,0


## 6. Converting and Handling Transaction Dates

The `Transaction Date` column was originally stored as text. Invalid and missing date values were converted to missing values and the column was then converted to the datetime data type.

Since the date column is important for time-based analysis, missing dates are filled using the most frequently occurring valid transaction date.

In [19]:
# Convert Transaction Date to datetime format
df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

# Fill missing dates with the mode
df["Transaction Date"] = df["Transaction Date"].fillna(
    df["Transaction Date"].mode()[0]
)

# Check the result
print(df["Transaction Date"].isnull().sum())
print(df["Transaction Date"].dtype)

0
datetime64[ns]


In [20]:
print("Missing Values After Cleaning:")
print(df.isnull().sum())

print("\nData Types After Cleaning:")
print(df.dtypes)

Missing Values After Cleaning:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

Data Types After Cleaning:
Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object


In [21]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,8.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


## 7. Duplicate Row Removal

Duplicate rows can cause the same transaction to be counted more than once and may lead to inaccurate analysis. The dataset was checked for exact duplicate rows.

If duplicate rows are found, they will be removed while keeping the first occurrence. In this dataset, the initial inspection found no exact duplicate rows.

In [22]:
# Count duplicate rows before removal
duplicate_count = df.duplicated().sum()

print("Duplicate rows found:", duplicate_count)

# Remove duplicate rows
df = df.drop_duplicates()

print("Dataset shape after duplicate removal:", df.shape)

Duplicate rows found: 0
Dataset shape after duplicate removal: (10000, 8)


## 8. Standardisation and Data Type Correction

The dataset was standardised to ensure consistent formatting and correct data types. Transaction IDs were converted to string format, numerical columns were converted to numeric types, and transaction dates were converted to datetime format.

In [23]:
# Convert Transaction ID to string
df["Transaction ID"] = df["Transaction ID"].astype("string")

# Ensure numeric columns have correct data types
df["Quantity"] = df["Quantity"].astype(float)
df["Price Per Unit"] = df["Price Per Unit"].astype(float)
df["Total Spent"] = df["Total Spent"].astype(float)

# Ensure Transaction Date is datetime
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])

# Display data types
df.dtypes

,0
Transaction ID,string[python]
Item,object
Quantity,float64
Price Per Unit,float64
Total Spent,float64
Payment Method,object
Location,object
Transaction Date,datetime64[ns]


## 9. Outlier Detection

The Interquartile Range (IQR) method was used to identify potential outliers in the numerical columns. The IQR method defines values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR as potential outliers.

Outliers were identified for review. Since unusual transactions may represent legitimate café purchases, they will be retained unless they are clearly invalid.

In [24]:
# Detect outliers using IQR method

outlier_summary = {}

for column in ["Quantity", "Price Per Unit", "Total Spent"]:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    outlier_summary[column] = len(outliers)

outlier_summary

{'Quantity': 0, 'Price Per Unit': 0, 'Total Spent': 259}

## 10. Before vs After Data Cleaning Summary

A comparison was created to evaluate the improvement in data quality after the cleaning process. The comparison includes missing values, duplicate rows, row count, and data type accuracy.

In [25]:
# Create before vs after summary

before_after_summary = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Missing Values",
        "Duplicate Rows"
    ],
    "Before Cleaning": [
        10000,
        6826,
        0
    ],
    "After Cleaning": [
        len(df),
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ]
})

before_after_summary

,Metric,Before Cleaning,After Cleaning
0,Total Rows,10000,10000
1,Total Missing Values,6826,0
2,Duplicate Rows,0,0


In [26]:
df.dtypes

,0
Transaction ID,string[python]
Item,object
Quantity,float64
Price Per Unit,float64
Total Spent,float64
Payment Method,object
Location,object
Transaction Date,datetime64[ns]


## 11. Save Cleaned Dataset

After completing the data cleaning process, the cleaned dataset is saved as a new CSV file. This preserves the original raw dataset and creates a separate analysis-ready version.

In [27]:
# Save the cleaned dataset as a new CSV file

df.to_csv("cleaned_dirty_cafe_sales.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [28]:
import os

os.listdir()

['.config',
 'dirty_cafe_sales.csv',
 'cleaned_dirty_cafe_sales.csv',
 'sample_data']